In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.environ['DATASET_25'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv'
os.environ['DATASET_15'] = '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv'

print(os.path.exists(os.environ['DATASET_25']))
print(os.path.exists(os.environ['DATASET_15']))

sys.path.append('/content')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
True
True


In [11]:
sys.path.append('/content/drive/MyDrive')

In [ ]:
import sys
sys.argv = [
    'data_accounting_20_seed__T11_.ipynb',
    '--csv25', '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv',
    '--csv15', '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv',
    '--out',   '/content/drive/MyDrive/Detector_Results/data_accounting_20seed'
]

# Jalankan script dari cell 3 (bukan dari file eksternal)
# Semua fungsi dan main() sudah didefinisikan di cell berikutnya
# Cukup jalankan cell 3 terlebih dahulu, lalu panggil main() di sini


In [ ]:
"""
Data Accounting — Multi-Seed Evaluation (n=20)
================================================
REVISED VERSION — 2 fixes dari versi asli:

FIX 1: count_windows_per_episode sekarang pakai step_sec=window_sec
        (NON-overlapping), sesuai dengan pipeline notebook asli.
        Versi asli pakai step_sec=1 (overlapping) yang menghasilkan
        jumlah window jauh lebih besar dari yang digunakan model.

FIX 2: argparse sekarang menerima --csv25 dan --csv15 terpisah.
        Sebelumnya hanya --csv sehingga 15 m/s selalu 0 karena
        Dataset_1.csv tidak punya data speed 13-17 m/s.

CARA MENJALANKAN di Colab:
    import sys
    sys.argv = [
        'data_accounting_20_seed__T11_.ipynb',
        '--csv25', '/content/drive/MyDrive/Dataset RF Jamming/Dataset_1.csv',
        '--csv15', '/content/drive/MyDrive/Dataset RF Jamming/Dataset_2.csv',
        '--out',   '/content/drive/MyDrive/Detector_Results/data_accounting_20seed'
    ]
    main()

    atau:
    %run data_accounting_20_seed__T11_.ipynb \
        --csv25 "/content/.../Dataset_1.csv" \
        --csv15 "/content/.../Dataset_2.csv" \
        --out   "/content/.../data_accounting_20seed"
"""

import argparse
import os
import numpy as np
import pandas as pd

# ============================================================================
# KONFIGURASI — HARUS PERSIS SAMA DENGAN NOTEBOOK FINAL
# ============================================================================

SEEDS = [42, 123, 456, 789, 2024, 7, 13, 21, 33, 55,
         77, 99, 111, 222, 333, 444, 555, 777, 888, 999]
CHECKPOINTS  = [5, 10, 15, 20]
WINDOW_SIZES = [1, 5, 15]          # detik
EPISODE_LENGTH = 60                 # detik per pseudo-episode bucket
FEATURES     = ['SNR', 'RSSI', 'PDR', 'Speed', 'Relative_Speed']
SPEED_RANGE  = {15: (13, 17), 25: (23, 27)}
SCENARIO_TO_LABEL = {1: 0, 2: 1, 3: 2}
CLASS_NAMES  = {0: 'Interference', 1: 'Reactive', 2: 'Constant'}
TRAIN_RATIO  = 0.6
VAL_RATIO    = 0.2


# ============================================================================
# FUNGSI RE-IMPLEMENTASI DARI NOTEBOOK
# ============================================================================

def build_pseudo_run_id(df):
    """Verbatim dari notebook Cell 2."""
    return (
        df['Speed'].round(2).astype(str) + '_' +
        df['Scenario'].astype(str) + '_' +
        (df['Time'] // EPISODE_LENGTH).astype(int).astype(str)
    )


def split_by_episode(df, seed, train_ratio=TRAIN_RATIO, val_ratio=VAL_RATIO):
    """Verbatim dari notebook Cell 3."""
    rng = np.random.RandomState(seed)
    ep_to_label = (
        df.groupby('pseudo_run_id')['label']
          .agg(lambda x: x.iloc[0])
          .to_dict()
    )
    eps_by_class = {}
    for ep, lbl in ep_to_label.items():
        eps_by_class.setdefault(lbl, []).append(ep)

    train_eps, val_eps, test_eps = [], [], []
    for lbl, eps in eps_by_class.items():
        eps = np.array(eps)
        rng.shuffle(eps)
        n = len(eps)
        n_train = int(round(train_ratio * n))
        n_val   = int(round(val_ratio * n))
        if n >= 3:
            n_train = max(1, min(n - 2, n_train))
            n_val   = max(1, min(n - n_train - 1, n_val))
        train_eps.extend(eps[:n_train].tolist())
        val_eps.extend(eps[n_train:n_train + n_val].tolist())
        test_eps.extend(eps[n_train + n_val:].tolist())

    return set(train_eps), set(val_eps), set(test_eps)


def count_windows_per_episode(times, window_sec):
    """
    FIX 1: step_sec = window_sec (NON-overlapping), verbatim notebook.

    Versi asli pakai step_sec=1 (overlapping) — itu bug.
    Notebook asli: t0 += step_sec di mana step_sec = window_sec.
    """
    if len(times) == 0:
        return 0
    step_sec = window_sec          # NON-overlapping, sesuai notebook
    t0    = float(times[0])
    t_end = float(times[-1])
    n = 0
    while t0 + window_sec <= t_end:
        mask = (times >= t0) & (times < t0 + window_sec)
        if mask.sum() > 0:
            n += 1
        t0 += step_sec
    return n


# ============================================================================
# TABLE BUILDERS
# ============================================================================

def table_a_global(df25_raw, df15_raw):
    """
    FIX 2: tiap regime pakai df dari file yang benar.
    Sebelumnya: df_raw = concat(df1+df2) -> filter 23-27 juga
                menangkap baris dari df2 (Dataset_2 punya 38 baris
                di speed 23-27 untuk Constant).
    Sekarang:   25 m/s -> df25_raw saja, 15 m/s -> df15_raw saja.
    """
    rows = []
    for target_speed, (lo, hi), df_raw in [
        (25, (23, 27), df25_raw),
        (15, (13, 17), df15_raw),
    ]:
        df_reg = df_raw[(df_raw['Speed'] >= lo) & (df_raw['Speed'] <= hi)].copy()
        for scen, lbl in SCENARIO_TO_LABEL.items():
            cls = CLASS_NAMES[lbl]
            raw_all  = int((df_raw['Scenario'] == scen).sum())
            retained = int((df_reg['Scenario'] == scen).sum())
            n_ep     = int(df_reg[df_reg['Scenario'] == scen]['pseudo_run_id'].nunique())
            rows.append({
                'regime_m_per_s': target_speed,
                'class_id': lbl, 'class_name': cls, 'scenario': scen,
                'raw_samples_all_speeds': raw_all,
                'retained_samples_in_regime': retained,
                'n_episodes': n_ep,
            })
    return pd.DataFrame(rows)


def table_b_episode_duration(df25_raw, df15_raw):
    """Distribusi durasi episode per (regime, class)."""
    rows = []
    for target_speed, (lo, hi), df_raw in [
        (25, (23, 27), df25_raw),
        (15, (13, 17), df15_raw),
    ]:
        df_reg = df_raw[(df_raw['Speed'] >= lo) & (df_raw['Speed'] <= hi)].copy()
        for lbl in [0, 1, 2]:
            sub = df_reg[df_reg['label'] == lbl]
            durations = (
                sub.groupby('pseudo_run_id')['Time']
                   .agg(lambda t: float(t.max() - t.min()))
                   .values
            )
            if len(durations) == 0:
                row = {
                    'regime_m_per_s': target_speed,
                    'class_id': lbl, 'class_name': CLASS_NAMES[lbl],
                    'n_episodes': 0,
                    'dur_min_s': np.nan, 'dur_median_s': np.nan,
                    'dur_mean_s': np.nan, 'dur_max_s': np.nan,
                    **{f'usable_for_w{w}s': 0 for w in WINDOW_SIZES},
                }
            else:
                row = {
                    'regime_m_per_s': target_speed,
                    'class_id': lbl, 'class_name': CLASS_NAMES[lbl],
                    'n_episodes': int(len(durations)),
                    'dur_min_s':    float(np.min(durations)),
                    'dur_median_s': float(np.median(durations)),
                    'dur_mean_s':   float(np.mean(durations)),
                    'dur_max_s':    float(np.max(durations)),
                    **{f'usable_for_w{w}s': int((durations >= w).sum())
                       for w in WINDOW_SIZES},
                }
            rows.append(row)
    return pd.DataFrame(rows)


def per_seed_and_windows(df_raw, target_speed):
    """Table C & D untuk satu regime."""
    lo, hi = SPEED_RANGE[target_speed]
    df_reg = df_raw[(df_raw['Speed'] >= lo) & (df_raw['Speed'] <= hi)].copy()

    ep_meta = {}
    for ep_id, grp in df_reg.groupby('pseudo_run_id'):
        grp_sorted = grp.sort_values('Time')
        ep_meta[ep_id] = {
            'label': int(grp_sorted['label'].iloc[0]),
            'times': grp_sorted['Time'].values,
        }

    rows_c, rows_d = [], []
    for seed in SEEDS:
        tr, va, te = split_by_episode(df_reg, seed=seed)

        # Table C: episode count per class per split
        row_c = {'regime_m_per_s': target_speed, 'seed': seed}
        for split_name, split_eps in [('train', tr), ('val', va), ('test', te)]:
            counts = {0: 0, 1: 0, 2: 0}
            for ep in split_eps:
                counts[ep_meta[ep]['label']] += 1
            for lbl in [0, 1, 2]:
                row_c[f'{split_name}_ep_class{lbl}_{CLASS_NAMES[lbl]}'] = counts[lbl]
            row_c[f'{split_name}_ep_total'] = sum(counts.values())
            row_c[f'{split_name}_class_complete'] = int(all(v > 0 for v in counts.values()))
        rows_c.append(row_c)

        # Table D: window count di TEST per class per window_size (NON-overlapping)
        for w in WINDOW_SIZES:
            win_counts  = {0: 0, 1: 0, 2: 0}
            usable_eps  = {0: 0, 1: 0, 2: 0}
            for ep in te:
                lbl   = ep_meta[ep]['label']
                n_win = count_windows_per_episode(ep_meta[ep]['times'], w)
                win_counts[lbl] += n_win
                if n_win > 0:
                    usable_eps[lbl] += 1
            row_d = {
                'regime_m_per_s': target_speed,
                'seed': seed,
                'window_size_s': w,
                **{f'test_windows_class{lbl}_{CLASS_NAMES[lbl]}': win_counts[lbl]
                   for lbl in [0, 1, 2]},
                **{f'test_usable_ep_class{lbl}_{CLASS_NAMES[lbl]}': usable_eps[lbl]
                   for lbl in [0, 1, 2]},
                'test_windows_total': sum(win_counts.values()),
                'test_window_class_complete': int(all(v > 0 for v in win_counts.values())),
            }
            rows_d.append(row_d)

    return pd.DataFrame(rows_c), pd.DataFrame(rows_d)


def table_e_checkpoints(df_c_25, df_d_25, df_c_15, df_d_15):
    """Agregat mean±std di checkpoint n=5/10/15/20."""
    rows = []
    for regime, df_c, df_d in [(25, df_c_25, df_d_25), (15, df_c_15, df_d_15)]:
        for n_ck in CHECKPOINTS:
            seeds_ck = SEEDS[:n_ck]
            sub_c = df_c[df_c['seed'].isin(seeds_ck)]
            sub_d = df_d[df_d['seed'].isin(seeds_ck)]
            for lbl in [0, 1, 2]:
                cls  = CLASS_NAMES[lbl]
                col  = f'test_ep_class{lbl}_{cls}'
                row  = {
                    'regime_m_per_s': regime,
                    'checkpoint_n_seeds': n_ck,
                    'class_id': lbl, 'class_name': cls,
                    'test_ep_mean': float(sub_c[col].mean()),
                    'test_ep_std':  float(sub_c[col].std(ddof=1)) if len(sub_c) > 1 else 0.0,
                    'test_ep_min':  int(sub_c[col].min()),
                }
                for w in WINDOW_SIZES:
                    sub_w = sub_d[sub_d['window_size_s'] == w]
                    wcol  = f'test_windows_class{lbl}_{cls}'
                    row[f'test_win_w{w}s_mean'] = float(sub_w[wcol].mean())
                    row[f'test_win_w{w}s_std']  = (
                        float(sub_w[wcol].std(ddof=1)) if len(sub_w) > 1 else 0.0)
                    row[f'test_win_w{w}s_min']  = int(sub_w[wcol].min())
                    row[f'test_win_w{w}s_seeds_with_zero'] = int((sub_w[wcol] == 0).sum())
                rows.append(row)
    return pd.DataFrame(rows)


# ============================================================================
# MAIN
# ============================================================================

def main():
    ap = argparse.ArgumentParser()
    # FIX 2: dua argumen CSV terpisah, satu per regime
    ap.add_argument('--csv25', required=True,
                    help='Path Dataset_1.csv (sumber 25 m/s)')
    ap.add_argument('--csv15', required=True,
                    help='Path Dataset_2.csv (sumber 15 m/s)')
    ap.add_argument('--out',   required=True,
                    help='Direktori output')
    args = ap.parse_args()

    os.makedirs(args.out, exist_ok=True)

    # Load per-regime dari file yang benar
    print(f'Loading 25 m/s: {args.csv25}')
    df25_raw = pd.read_csv(args.csv25)
    df25_raw['label'] = df25_raw['Scenario'].map(SCENARIO_TO_LABEL)
    df25_raw = df25_raw.sort_values(['Speed','Scenario','Time']).reset_index(drop=True)
    df25_raw['pseudo_run_id'] = build_pseudo_run_id(df25_raw)
    print(f'  Rows: {len(df25_raw):,}')

    print(f'Loading 15 m/s: {args.csv15}')
    df15_raw = pd.read_csv(args.csv15)
    df15_raw['label'] = df15_raw['Scenario'].map(SCENARIO_TO_LABEL)
    df15_raw = df15_raw.sort_values(['Speed','Scenario','Time']).reset_index(drop=True)
    df15_raw['pseudo_run_id'] = build_pseudo_run_id(df15_raw)
    print(f'  Rows: {len(df15_raw):,}')

    # TABLE A
    print('\n[Table A] Global accounting per (regime, class)...')
    df_a = table_a_global(df25_raw, df15_raw)
    df_a.to_csv(os.path.join(args.out, 'accounting_global.csv'), index=False)
    print(df_a.to_string(index=False))

    # TABLE B
    print('\n[Table B] Episode duration distribution + usability per window_size...')
    df_b = table_b_episode_duration(df25_raw, df15_raw)
    df_b.to_csv(os.path.join(args.out, 'accounting_episode_duration.csv'), index=False)
    print(df_b.to_string(index=False))

    # TABLE C + D per regime
    print('\n[Table C+D] Per-seed accounting (25 m/s)...')
    df_c_25, df_d_25 = per_seed_and_windows(df25_raw, 25)
    df_c_25.to_csv(os.path.join(args.out, 'accounting_per_seed_25ms.csv'), index=False)
    df_d_25.to_csv(os.path.join(args.out, 'accounting_test_windows_25ms.csv'), index=False)

    print('\n[Table C+D] Per-seed accounting (15 m/s)...')
    df_c_15, df_d_15 = per_seed_and_windows(df15_raw, 15)
    df_c_15.to_csv(os.path.join(args.out, 'accounting_per_seed_15ms.csv'), index=False)
    df_d_15.to_csv(os.path.join(args.out, 'accounting_test_windows_15ms.csv'), index=False)

    # TABLE E
    print('\n[Table E] Checkpoint aggregation n=5/10/15/20...')
    df_e = table_e_checkpoints(df_c_25, df_d_25, df_c_15, df_d_15)
    df_e.to_csv(os.path.join(args.out, 'accounting_checkpoints.csv'), index=False)
    print(df_e.to_string(index=False))

    # DIAGNOSTIC WARNINGS
    print('\n' + '=' * 70)
    print('DIAGNOSTIC WARNINGS')
    print('=' * 70)

    for regime, df_c in [(25, df_c_25), (15, df_c_15)]:
        incomplete = df_c[df_c['test_class_complete'] == 0]
        if len(incomplete) > 0:
            print(f'\n[{regime} m/s] {len(incomplete)}/20 seed test fold TIDAK class-complete (episode-level):')
            ep_cols = ['seed'] + [c for c in df_c.columns if c.startswith('test_ep_class')]
            print(incomplete[ep_cols].to_string(index=False))
        else:
            print(f'\n[{regime} m/s] Semua 20 seed class-complete di episode-level.')

    for regime, df_d in [(25, df_d_25), (15, df_d_15)]:
        for w in WINDOW_SIZES:
            sub = df_d[(df_d['window_size_s'] == w) &
                       (df_d['test_window_class_complete'] == 0)]
            if len(sub) > 0:
                print(f'\n[{regime} m/s, window={w}s] {len(sub)}/20 seed ada kelas tanpa test window:')
                wcols = ['seed'] + [c for c in df_d.columns if c.startswith('test_windows_class')]
                print(sub[wcols].to_string(index=False))

    print(f'\nSelesai. Output di: {args.out}')


if __name__ == '__main__':
    main()

In [ ]:
# Jalankan setelah cell 3 selesai dieksekusi
main()